<a href="https://colab.research.google.com/github/samsarkar184/Self-Supervised-Learning/blob/main/SimCLR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#imports
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader,Subset
from torchvision import transforms,datasets
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [3]:
#gpu check
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
#transformation of image
train_transform=transforms.Compose([transforms.RandomHorizontalFlip(),
                                    transforms.RandomResizedCrop(32,scale=(0.5,1.0)),
                                    transforms.RandomApply([
                                        transforms.ColorJitter(brightness=0.8,contrast=0.8,saturation=0.8,hue=0.2)
                                    ],p=0.8),
                                    transforms.ToTensor()])

In [ ]:
#import CIFAR-10 train dataset
train_dataset=datasets.CIFAR10(root="./data",
                               train=True,
                               download=True,
                               transform=None)

 36%|███▌      | 60.7M/170M [13:57<29:49, 61.4kB/s]

In [1]:
#import CIFAR-10 test dataset
transform=transforms.ToTensor()
test_dataset=datasets.CIFAR10(root="./data",
                              train=False,
                              download=True,
                              transform=transform)

NameError: name 'transforms' is not defined

In [ ]:
#take a subset of the dataset for training and testing
train_subset=Subset(train_dataset,range(5000))
test_subset=Subset(test_dataset,range(1000))
train_loader=DataLoader(train_subset,batch_size=128,shuffle=True)
test_loader=DataLoader(test_subset,batch_size=128,shuffle=True)

In [ ]:
# creating SimClr wrapper
from torch.utils.data import Dataset
class Simdataset(Dataset):
    def __init__(self,dataset,transform):
        self.dataset = dataset
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        image = self.dataset[index][0]
        view_1 = self.transform(image)
        view_2 = self.transform(image)
        return view_1, view_2

In [ ]:
# creating object of simdataset
sim_dataset = Simdataset(train_subset, train_transform)

In [ ]:
# displaying the no. of samples and two augmented view
print("Number of samples:", len(sim_dataset))

view_1, view_2 = sim_dataset[0]

print("View 1 shape:", view_1.shape)
print("View 2 shape:", view_2.shape)

In [ ]:
# creating SimClr DataLoader
from torch.utils.data import DataLoader
sim_loader = DataLoader(
    sim_dataset,
    batch_size = 128,
    shuffle = True,
    drop_last = True
)

In [ ]:
# Testing the DataLoader
view_1_batch, view_2_batch = next(iter(sim_loader))
print("View 1 batch shape:", view_1_batch.shape)
print("View 2 batch shape:", view_2_batch.shape)

In [ ]:
class Encoder(nn.Module):
  def __init__(self):
    super(Encoder,self).__init__()
    self.conv1=nn.Conv2d(3,32,kernel_size=3,padding=1)
    self.conv2=nn.Conv2d(32,64,kernel_size=3,padding=1)
    self.pool=nn.Maxpool2d(2,2)
    self.fc=nn.Linear(64*8*8,128)

  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=self.pool(x)
    x=F.relu(self.conv2(x))
    x=self.pool(x)
    x=torch.flatten(x,1)
    x=self.fc(x)
    return x
